In [ ]:
import os

#Navigate to the new os environment containing the Pandeia engine and its packages

os.environ['pandeia_refdata'] = "C:/Users/15125/Downloads/pandeia_data-4.0-jwst"
os.environ['PYSYN_CDBS'] = "C:/Users/15125/Downloads/grp/redcat/trds"

In [ ]:
#Import required libraries

import numpy as np
import matplotlib.pyplot as plt
import random
import astropy
from astropy.table import Table
import time

In [ ]:
%matplotlib inline

In [ ]:
#Import a default Pandeia build with our telescope/filter specifics
from pandeia.engine.calc_utils import build_default_calc

calculation0 = build_default_calc("jwst", "nirspec", "mos")

In [ ]:
calculation = build_default_calc("jwst", "nirspec", "mos")

In [ ]:
#conduct spectral calculation based on the original default build
from pandeia.engine.perform_calculation import perform_calculation

In [ ]:
calculation0

In [ ]:
#read in a basic flat array to conduct calibration
wave_in_um = np.arange(.5,6,.01)
flux_in_mJy = 1e3*np.ones_like(wave_in_um)
clight= 2.99792458E+18 #in angstrom/s

In [ ]:
#change the deafult build to fit match my data specifications properly

calculation0['configuration']['detector']={'nexp':3,'ngroup':65,'nint':1,'readout_pattern':'nrsirs2rapid','subarray':'full'}
calculation0['background_level']='low'
for i in ['crs','dark','excess','ffnoise','readnoise','scatter']: calculation0['calculation']['noise'][i]=True
calculation0['scene'][0]['spectrum']['normalization']={'type':'none'}
calculation0['scene'][0]['spectrum']['redshift']=0 #included redshift in the spectrum itself
calculation0['scene'][0]['shape']={'geometry':'sersic','major':0.07,'minor':0.07,'sersic_index':1.0,'norm_method':'integ_infinity','surf_area_units':None}
calculation0['scene'][0]['spectrum']['sed']={'sed_type':'input','spectrum':[wave_in_um,flux_in_mJy]}
calculation0['configuration']['instrument']['disperser']='prism'
calculation0['configuration']['instrument']['filter']= 'clear'
report0=perform_calculation(calculation0)

#This will serve as my calibration, I will divide this from each point in my generated spectra to convert flux from counts into Jansky

#EVENTUALLY, I WANT TO CREATE A COMPLETENESS PLOT AS A 2D FUNCTION OF FLUX AND REDSHIFT, WITH COMPLETENESS BEING COLOR-CODED

In [ ]:
#Test printout
min(report0['1d']['extracted_flux'][0]),max(report0['1d']['extracted_flux'][0])

In [ ]:
#Display calibration curve to divide my output spectra by
plt.figure(dpi=150)
plt.plot(report0['1d']['extracted_flux'][0], report0['1d']['extracted_flux'][1])
plt.xlabel("Wavelength (Microns)")
plt.ylabel("Extracted Flux (Counts/Jansky)")
plt.title("Pandeia Calibration Calculated Counts")
plt.show()

In [ ]:
#Test output
wave_cal = report0['1d']['extracted_flux'][0]
flux_cal = report0['1d']['extracted_flux'][1]

In [ ]:
#redshift, flux, FWHM, EW (continuum), post-data specifications
report0

In [ ]:
#units in counts/pixel

In [ ]:
#Now, we try to simulate a spectra, allowing the user to enter information on redshift, flux, FWHM
#FWHM = 2 * sqrt(2ln(2)) * σ from Gaussian distribution
#H-alpha is at 6562.8 angstrom, with a fwhm of typically 60nm (or .06 micron)
#h_alpha_mu = [6562.8*(z+1)]
#int_fluxes_emcee = Amp * sigma * np.sqrt(2pi)


#USER INPUT VERSION
# z = input("Enter Redshift: ")
# fwhm = input("Enter FWHM for Line Width (in microns): ")
# flx = input("Enter Integrated Flux Value (in Jy): ")


#SINGLE TEST CASE VERSION
# z = 4.3
# fwhm = 0.08
# flx = 1.33e-17

#RANDOM GENERATOR VERSION
rand_z = random.uniform(2,7) #go by .5
#rand_fwhm = random.uniform(10, 100)  #Max and min are from sample (in angstrom)
rand_flx = 10**random.uniform(np.log10(2.7e-19), np.log10(3.3e-16)) #Max and min are from sample! go from 1e-19 to 1e-16 or 10**(np.arange(-19,-15.9, 0.1))


# lsts = 10**np.arange(-19, -15.9, .1)
# pick = random.randint(1,31)
# rand_flx = lsts[pick]


rand_fwhm = 10

# rand_z=4
# rand_fwhm=20
# rand_flx=1e-18


fac = 2*np.sqrt(2*np.log(2))   #from equation of FWHM in relation to sigma from a Gaussian distribution
count = 0

#Simulate a spectra based on a random redshift, width, and amplitude within the appropriate ranges
def sim_spectra(z,fwhm,flx):
    
    num = count+1
    sigma = (1+float(z))*(float(fwhm)/fac)                      #get the variables for the spectrum
    mu = .65628*1e4*(float(z)+1)                     #mu wavelength is in angstrom
    amp = flx/(sigma*np.sqrt(2*np.pi)) 
    print("Randomly generating redshift, FWHM, and integrated flux...")
    print("Redshift: " + str(z) + ", FWHM: " + str(fwhm) + ", Integrated Flux: " + str(flx))
    print("Amp: " + str(amp) + ", Mu: " + str(mu) + ", Sigma: " + str(sigma))
    
    waves = np.arange(1e4*.5,1e4*6,.0025*1e4)
    fluxes=[]
    
    for i in waves:
        fluxes.append((float(flx)/(sigma*np.sqrt(2*np.pi)))*np.exp(-0.5*(((i-mu)/sigma)**2)))
  #  print("Initial flux array: " + str(fluxes))

    
    #NOISE SIMULATION, ENDED UP BEING UNNECESSARY, PANDEIA WILL DO THIS FOR ME
    # big = max(fluxes)
    # for i in range(len(fluxes)):
    #     if np.abs(waves[i] - mu) > .25:
    #          noise_fac = random.uniform(-.1, .1)
    #          noise = noise_fac*big
    #          fluxes[i] = noise+fluxes[i]

    #Plot initial simulated spectra
    # pic = plt.figure()
    # plt.plot(waves, fluxes)
    # plt.xlabel("Wavelength(Angstrom)")
    # plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
    # plt.title("Initial Simulated Image " + str(num) + " (z = " + str(z) + ")")
    # plt.show()


    # print(len(waves), len(fluxes))
    # print(len(report['1d']['extracted_flux'][0]), len(report['1d']['extracted_flux'][1]))
    # print(waves, 1e21*fluxes*((waves**2)/clight))
    # print(fluxes)

  
    #Completing the unit conversions
    fluxes_arr_nu = (fluxes*((waves**2)/clight))*1e23 #convert to F_nu units (Jy)
  #  print("Fluxes in f_nu units: " + str(fluxes_arr_nu))
    fluxes_arr_mJy = 1e3*fluxes_arr_nu #convert to mJy fully
    
 #   print(fluxes_arr_mJy)
    waves_arr_um = 1e-4*waves
#    print(waves, waves_arr_um)

    # pic2 = plt.figure()
    # plt.plot(waves_arr_um, fluxes_arr_mJy)
    # plt.xlabel("Wavelength(Micron)")
    # plt.ylabel("Flux (mJy)")
    # plt.title("Pre-Pandeia Simulated Image " + str(num) + " (z = " + str(z) + ")")
    # plt.show()
    
    # print("Pandeia input wavelength and flux arrays")
    # print(waves_arr_um)
    # print(fluxes_arr_mJy)

    #Pandeia portion! Specifies PRISM disperser, CLEAR filter, input spectra, mos, etc.
    calculation['configuration']['detector']={'nexp':3,'ngroup':65,'nint':1,'readout_pattern':'nrsirs2rapid','subarray':'full'}
    calculation['background_level']='low'
    for i in ['crs','dark','excess','ffnoise','readnoise','scatter']: calculation['calculation']['noise'][i]=True
    calculation['scene'][0]['spectrum']['normalization']={'type':'none'}
    calculation['scene'][0]['spectrum']['redshift']=0 #included redshift in the spectrum itself
    calculation['scene'][0]['shape']={'geometry':'sersic','major':0.07,'minor':0.07,'sersic_index':1.0,'norm_method':'integ_infinity','surf_area_units':None}
    calculation['scene'][0]['spectrum']['sed']={'sed_type':'input','spectrum':[waves_arr_um,fluxes_arr_mJy]}
    calculation['configuration']['instrument']['disperser']='prism'
    calculation['configuration']['instrument']['filter']= 'clear'
    report=perform_calculation(calculation)

  #  print(report['1d']['extracted_flux'][1], report['1d']['extracted_noise'][1])
  #  print("Post-pandeia fluxes:" + str(report['1d']['extracted_flux'][1]))
    #Apply the calibration curve by dividing by it: Should fix units!
    use_waves = report['1d']['extracted_flux'][0]
    use_fluxes = report['1d']['extracted_flux'][1]/flux_cal
    use_fluxerrs = report['1d']['extracted_noise'][1]/flux_cal

    #Randomizing the seed so that Pandeia doesn't auto reset it to the same randomization every time
    np.random.seed(int(time.time()*100)%123456789)
    #Generating noise from the calibrated extracted_noise
    noise_arr = np.random.normal(0, use_fluxerrs)
    noisy_fluxes = use_fluxes + noise_arr


    # pic3 = plt.figure()
    # plt.plot(use_waves, report['1d']['extracted_flux'][1], color = 'green', label = 'Post-Pandeia')
    # plt.xlabel("Wavelength (Micron)")
    # plt.ylabel("Flux (Counts)")
    # plt.title("Post-Pandeia Simulated Image " + str(num) + " (z = " + str(z) + ")")
    # plt.show()

    # pic4 = plt.figure()
    # plt.plot(use_waves, use_fluxes, color = 'red', label = 'Post-Calibration Division')
    # plt.xlabel("Wavelength (Micron)")
    # plt.ylabel("Flux (Jy)")
    # plt.title("Post-Pandeia Calibrated Image " + str(num) + " (z = " + str(z) + ")")
    # plt.show()

    pic5 = plt.figure(dpi=150)
    plt.plot(use_waves, noisy_fluxes, color = 'red', label = 'Post-Calibration Division')
   # plt.plot(waves_arr_um, 1e-3*fluxes_arr_mJy, color = "green")
    plt.xlabel("Wavelength (Micron)")
    plt.ylabel("Flux (Jy)")
    plt.title("Simulation Grid Image @" + " (z = " + str(z) + ") & Flux = " + str(flx))
    plt.axvline(1e-4*mu, color = "blue")
 #   plt.xlim(1e-4*mu-0.5,1e-4*mu+0.5)
    plt.show()

    
  #  Plot simulated spectra after running Pandeia    
    # pic6 = plt.figure()
    # plt.plot(waves_arr_um, fluxes_arr_mJy/1000, label = "Original Simulated")
    # plt.plot(use_waves, report['1d']['extracted_flux'][1], color = 'green', label = 'Post-Pandeia')
    # plt.plot(use_waves, use_fluxes, label = "After dividing", color = 'red')
    # plt.xlabel("Wavelength(um)")
    # plt.ylabel("Flux (mJy)")
    # plt.title("Pics 2+3+4 for Simulated Image " + str(num) + "(z = " + str(z) + ")")
    # plt.legend()
    # plt.show()

    
    return(pic5, use_waves, use_fluxes, use_fluxerrs, sigma, mu, amp)
    
sim_spectra(rand_z,rand_fwhm,rand_flx)

In [ ]:
time.time()

In [ ]:
#feed gaussian into pandeia (mJy and micron), pandeia will output array in counts, divide by calibration array to get jansky#
#np.random.norm for applying errors to this after

In [ ]:
#Simulate a grid of spectra across the redshift and luminosity ranges of my sample
#Loop it for however many spectra you want!

redshift_range= np.arange(2,7.5,.5) #11 bins
flx_range = 10**np.arange(-19,-15.9,0.1) # x31 bins

#num_runs = 31 #number of spectra we will create

zs=[]
fwhms=[]
in_fluxes=[]
amps=[]
mus=[]
sigmas=[]
first_pics=[]
pics=[]
waves=[]
fluxes=[]
fluxerrs=[]

count = 0
rand_fwhm =10 # random.uniform(80,600)  #Max and min are from sample (in angstrom)
#while count < num_runs:
for z in redshift_range:
    rand_z = z #random.uniform(2,7)
    for flx in flx_range:
   # rand_flx = 10**random.uniform(np.log10(2.7e-19), np.log10(3.3e-16)) #Max and min are from sample! 
        rand_flx = flx

        pic5,wave, flux, fluxerr,sigma,mu,amp = sim_spectra(rand_z, rand_fwhm, rand_flx)
    
        zs.append(rand_z)
        fwhms.append(rand_fwhm)
        in_fluxes.append(rand_flx)
        
        sigmas.append(sigma)
        mus.append(mu)
        amps.append(amp)
        pics.append(pic5)
        #first_pics.append(pic)
    
        waves.append(wave)
        fluxes.append(flux)
        fluxerrs.append(fluxerr)
        
    
  #  count+=1

sim_grid = Table([pics, waves, fluxes, fluxerrs, zs, in_fluxes, fwhms, amps,mus, sigmas], names = ('Simulated Images', "Wavelength Arrays", "Flux Arrays", "Noise Arrays", 'Redshifts', 'Integrated Fluxes', 'FWHMs', 'Amplitudes', 'Mus', 'Sigmas'))
sim_grid
for i in range(len(sim_grid)):
    string = "grid" + str(i)
    pics[i].savefig(string)

In [ ]:
#Save the noise arrays for each simulation in the grid...these will be iterated through using time.time to vary noise randomly

flux_grid = np.array(sim_grid['Flux Arrays'])
np.savetxt("flux_grid.txt", flux_grid)
noise_grid = np.array(sim_grid['Noise Arrays'])
np.savetxt("noise_grid.txt", noise_grid)

In [ ]:
#Save grid of output values of the simulated Gaussian parameters for each emission line
sim_grid341 = Table([waves, fluxes, fluxerrs, zs, in_fluxes, fwhms, amps,mus, sigmas], names = ("Wavelength Arrays", "Flux Arrays", "Noise Arrays", 'Redshifts', 'Integrated Fluxes', 'FWHMs', 'Amplitudes', 'Mus', 'Sigmas'))
sim_grid341.write("sim_grid.fits")

In [ ]:
#Save the grid of simulated information
sim_grid = Table.read("sim_grid.fits")
sim_grid

In [ ]:
#This is to create an animation of one of the redshift bins as a function of luminosity increasing across the grid
#Saving frames...
import cv2
import glob

img_array = []
for filename in vidfiles:
    print(filename)
    img = cv2.imread(filename)
    height, width, layers = img.shape
    img_array.append(img)

out = cv2.VideoWriter("animation.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 5, (width, height))
for img in img_array:
    out.write(img)
out.release()


In [ ]:
#Reordering frames correctly
vidfiles =['vid0.png',
 'vid1.png',
 'vid2.png',
 'vid3.png',
 'vid4.png',
 'vid5.png',
 'vid6.png',
 'vid7.png',
 'vid8.png',
 'vid9.png',
 'vid10.png',
 'vid11.png',
 'vid12.png',
 'vid13.png',
 'vid14.png',
 'vid15.png',
 'vid16.png',
 'vid17.png',
 'vid18.png',
 'vid19.png',
 'vid20.png',
 'vid21.png',
 'vid22.png',
 'vid23.png',
 'vid24.png',
 'vid25.png',
 'vid26.png',
 'vid27.png',
 'vid28.png',
 'vid29.png',
 'vid30.png',]